# Uncertainty and calibration

Separate predictive confidence, entropy and empirical correctness.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Synthetic logits

Construct a deliberately overconfident classifier to isolate calibration behavior from model training.

In [ ]:
import matplotlib.pyplot as plt
import torch

from autonomous_vehicles.experiments.reproducibility import reproducible
from autonomous_vehicles.perception.calibration import (
    calibration_report,
    fit_temperature,
    predictive_entropy,
)
from autonomous_vehicles.visualization import plot_calibration

with reproducible(42):
    validation_logits = torch.randn(4, 3, 16, 16) * 4
    validation_labels = torch.randint(0, 3, (4, 16, 16))
    test_logits = torch.randn(4, 3, 16, 16) * 4
    test_labels = torch.randint(0, 3, (4, 16, 16))

## Before calibration

Confidence is a model output. Accuracy requires labels and may differ substantially.

In [ ]:
before = calibration_report(test_logits, test_labels)
print(before)
plot_calibration(before)
plt.show()
assert sum(item.count for item in before.bins) == test_labels.numel()

## Fit on validation only

Temperature is selected using validation negative log likelihood; test labels are kept out of fitting.

In [ ]:
with reproducible(42):
    temperature = fit_temperature(validation_logits, validation_labels, steps=40)
after = calibration_report(test_logits / temperature, test_labels)
print("Temperature:", temperature)
print("Before test NLL:", before.negative_log_likelihood)
print("After test NLL:", after.negative_log_likelihood)
plot_calibration(after)
plt.show()

## Entropy maps

Entropy summarizes the probability vector. It does not measure epistemic uncertainty by itself.

In [ ]:
original_entropy = predictive_entropy(test_logits)
calibrated_entropy = predictive_entropy(test_logits / temperature)
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(original_entropy[0], vmin=0, vmax=1.1)
axes[0].set_title("Original entropy")
axes[1].imshow(calibrated_entropy[0], vmin=0, vmax=1.1)
axes[1].set_title("Calibrated entropy")
plt.show()
assert torch.equal(test_logits.argmax(1), (test_logits / temperature).argmax(1))

## Confidence histogram

Scalar temperature changes confidence without changing the argmax class.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
for logits, name in ((test_logits, "Before"), (test_logits / temperature, "After")):
    confidence = logits.softmax(1).max(1).values.flatten().numpy()
    ax.hist(confidence, bins=15, alpha=0.5, label=name)
ax.set_xlabel("Top-class confidence")
ax.set_ylabel("Pixels")
ax.legend()
plt.show()
print("Calibration does not repair incorrect labels or domain shift.")

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.